# Preprocesamiento, Construcción de Splits y Entrenamiento de Modelos
## Tesis: Benchmarking Explainable Gradient Boosting and Tabular Deep Learning  
## for Predicting Satisfaction with Democracy in Latin America (1995–2024)

**Autor:** Mario Patricio Porras Martínez  
**Universidad:** Yachay Tech — Maestría en Inteligencia Artificial

---

### Propósito de este notebook

Este notebook implementa el **pipeline completo de preprocesamiento, partición temporal y entrenamiento** de los cinco modelos comparativos de la tesis. Parte de los archivos consolidados generados por el notebook `01_carga_datos.ipynb` y produce:

1. **Integración y fusión** de Latinobarómetro con V-Dem por clave `(pais_iso3, año)`.
2. **Corrección de anomalías** en variables de identificación temporal (`X_002`).
3. **Recodificaciones y transformaciones** de escala según la Opción B (mayor = más favorable).
4. **Imputación diferenciada** por tipo de modelo.
5. **Tres splits temporales** bajo el esquema *Expanding Window Walk-Forward Validation*.
6. **Entrenamiento, optimización de hiperparámetros y evaluación** de OLO, XGBoost, CatBoost, LightGBM y TabNet para cada subperiodo.
7. **Persistencia** de modelos entrenados y datasets procesados en formato Parquet.

### Esquema de subperiodos (Expanding Window)

```
SP1: train = 1995–2006 (11 olas) → test = 2007
SP2: train = 1995–2017 (20 olas) → test = 2018
SP3: train = 1995–2023 (23 olas) → test = 2024
```

> **Nota de reproducibilidad:** Semilla global fijada en `SEMILLA = 42`. Todas las versiones de librerías se registran al final del notebook.

## 0. Instalación de dependencias

Ejecutar una sola vez al configurar el entorno virtual.

In [ ]:
# =============================================================================
# CELDA 0 — Instalación de dependencias
# Ejecutar una sola vez en el entorno virtual (.venv)
# =============================================================================
import subprocess, sys

paquetes = [
    "pandas>=2.1.0",
    "numpy>=1.26.0",
    "scikit-learn>=1.4.0",
    "xgboost>=2.0.0",
    "lightgbm>=4.3.0",
    "catboost>=1.2.0",
    "pytorch-tabnet>=4.1.0",
    "optuna>=3.5.0",
    "imbalanced-learn>=0.12.0",
    "pyarrow>=14.0.0",       # requerido para leer/escribir Parquet
    "joblib>=1.3.0",
    "tqdm>=4.66.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
]

# for paquete in paquetes:
#    subprocess.check_call([sys.executable, "-m", "pip", "install", paquete, "-q"])

print("✓ Dependencias instaladas correctamente.")

## 1. Importaciones y configuración global

In [ ]:
# =============================================================================
# CELDA 1 — Importaciones globales
# =============================================================================
import os
import gc
import json
import joblib
import warnings
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy import stats

# Scikit-learn
from sklearn.experimental import enable_iterative_imputer   # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OrdinalEncoder, LabelEncoder
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    balanced_accuracy_score, f1_score,
    cohen_kappa_score, roc_auc_score,
    mean_absolute_error, confusion_matrix,
    classification_report
)
from sklearn.pipeline import Pipeline

# Modelos
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

# Optimización de hiperparámetros
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Progreso
from tqdm.notebook import tqdm

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(level=logging.WARNING)

print("✓ Importaciones completadas.")

## 2. Parámetros globales del pipeline

> **Todas las configuraciones ajustables se concentran en esta sección.**  
> Modificar aquí para cambiar rutas, hiperparámetros base, estrategia de CV o flags de ejecución.

In [ ]:
# =============================================================================
# CELDA 2 — PARÁMETROS GLOBALES DEL PIPELINE
# Modificar este bloque para ajustar cualquier configuración sin tocar
# el resto del notebook.
# =============================================================================

# ── Reproducibilidad ─────────────────────────────────────────────────────────
SEMILLA = 42
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# ── Hardware ──────────────────────────────────────────────────────────────────
USAR_GPU = False          # True si hay GPU CUDA disponible para TabNet y XGBoost
DISPOSITIVO_TABNET = "cuda" if USAR_GPU and torch.cuda.is_available() else "cpu"
N_JOBS = -1               # Número de núcleos CPU (-1 = todos)

# ── Rutas del proyecto ────────────────────────────────────────────────────────
# Ajustar RAIZ_PROYECTO si el notebook se ejecuta desde un directorio diferente
RAIZ_PROYECTO   = Path("..")
CARPETA_DATA    = RAIZ_PROYECTO / "data"
CARPETA_PROC    = CARPETA_DATA  / "processed"
CARPETA_LOADED    = CARPETA_DATA  / "loaded"
CARPETA_MODELS  = RAIZ_PROYECTO / "models"
CARPETA_RESULTS = RAIZ_PROYECTO / "results"

for carpeta in [CARPETA_PROC, CARPETA_MODELS, CARPETA_RESULTS]:
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO_LB   = CARPETA_LOADED / "latinobarometro_muestra.csv" # "latinobarometro.csv"
ARCHIVO_VDEM = CARPETA_LOADED / "v-dem.csv"

# ── Flags de ejecución ────────────────────────────────────────────────────────
# Poner en True para re-ejecutar búsqueda de hiperparámetros (tarda horas).
# False carga los hiperparámetros ya guardados si existen.
EJECUTAR_BUSQUEDA_HP = True

# Número de trials de Optuna por modelo y subperiodo
N_TRIALS_OPTUNA = 50      # Reducir a 20 para pruebas rápidas

# ── Definición de subperíodos (Expanding Window) ──────────────────────────────
SUBPERIODOS = {
    "SP1": {
        "train_olas": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006],
        "test_ola"  : [2007],
        "descripcion": "Consolidación democrática (1995–2007)",
    },
    "SP2": {
        "train_olas": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,
                        2007,2008,2009,2010,2011,2013,2015,2016,2017],
        "test_ola"  : [2018],
        "descripcion": "Crisis y reconfiguración (2008–2018)",
    },
    "SP3": {
        "train_olas": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,
                        2007,2008,2009,2010,2011,2013,2015,2016,2017,2018,2020,2023],
        "test_ola"  : [2024],
        "descripcion": "Pandemia y recuperación (2020–2024)",
    },
}

# ── Validación cruzada temporal ────────────────────────────────────────────────
N_SPLITS_CV = 5   # TimeSeriesSplit con 5 folds sobre el conjunto de train

# ── Definición de subregiones ─────────────────────────────────────────────────
# Para el análisis de estabilidad regional (OE3)
SUBREGIONES = {
    "Cono Sur"        : ["Argentina", "Chile", "Uruguay", "Paraguay", "Perú"],
    "Región Andina"   : ["Bolivia", "Colombia", "Ecuador", "Venezuela"],
    "Brasil"          : ["Brasil"],
    "Centroamérica"   : ["Costa Rica", "El Salvador", "Guatemala",
                          "Honduras", "Nicaragua", "Panamá"],
    "México y Caribe" : ["México", "República Dominicana"],
}

# ── Tratamiento especial de países ────────────────────────────────────────────
# Venezuela: incluir en train hasta 2017, excluir del test y análisis por país
# Nicaragua: incluir en train completo, excluir del análisis por país en test
PAISES_EXCLUIR_TEST_REGIONAL = ["Venezuela", "Nicaragua"]
AÑO_CORTE_VENEZUELA = 2017   # incluir en train solo hasta este año

# ── Mapeo de numinves → año calendario ────────────────────────────────────────
# X_002 en algunas olas contiene el número de investigación, no el año
MAPEO_NUMINVES_AÑO = {
    16: 2011,
    17: 2013,
    18: 2015,
    23: 2023,
    24: 2024,
}

# ── Mapeo pais_nombre → ISO3 ──────────────────────────────────────────────────
MAPEO_PAIS_ISO3 = {
    "Argentina"           : "ARG",
    "Bolivia"             : "BOL",
    "Brasil"              : "BRA",
    "Chile"               : "CHL",
    "Colombia"            : "COL",
    "Costa Rica"          : "CRI",
    "República Dominicana": "DOM",
    "Ecuador"             : "ECU",
    "El Salvador"         : "SLV",
    "Guatemala"           : "GTM",
    "Honduras"            : "HND",
    "México"              : "MEX",
    "Nicaragua"           : "NIC",
    "Panamá"              : "PAN",
    "Paraguay"            : "PRY",
    "Perú"                : "PER",
    "Uruguay"             : "URY",
    "Venezuela"           : "VEN",
}

# ── Variable dependiente ──────────────────────────────────────────────────────
COL_TARGET = "target"          # nombre final de la VD (base 0: {0,1,2,3})
COL_AÑO    = "año"             # columna de año calendario tras corrección
COL_PAIS   = "pais_nombre"     # columna de nombre de país
COL_ISO3   = "pais_iso3"       # columna ISO3 (derivada)
COL_PESO   = "X_020"           # factor de ponderación muestral

# ── Códigos NS/NR a convertir a NaN ───────────────────────────────────────────
# Todos los valores negativos en LB son NS/NR excepto donde se documente lo contrario
NSNR_GENERICOS = [-1, -2, -3, -4, -5, -6, -7, -8]

# ── Variables EXCLUIDAS del dataset final (justificación en metodología) ──────
VARS_EXCLUIR_LB = [
    "C_001_031",    # ruptura de codificación en 2018; correlación ≈ 0
    "A_003_021",    # ausente en SP2_test y SP3_test; distorsiona SHAP
    "D_001_061",    # ausente en los 3 conjuntos de prueba
    "D_001_131",    # ausente en SP2_test y SP3_test
    "X_004",        # 627 categorías únicas, 94% nuevas en SP3_test; sin señal
]

VARS_EXCLUIR_VDEM = [
    "v2x_neopat",       # r=0.970 con v2x_rule; redundancia crítica
    "v2xnp_regcorr",    # r=0.985 con v2x_execorr; redundancia crítica
    "v2xpe_exlsocgr",   # NaN en todos los países en 2024 (SP3_test)
    "v2xpe_exlecon",    # NaN en todos los países en 2024 (SP3_test)
]

# ── Variables V-Dem con dirección INVERTIDA ───────────────────────────────────
# Mayor valor = PEOR situación democrática → aplicar transformación 1 - x
VARS_VDEM_INVERTIR = [
    "v2x_corr",
    "v2x_execorr",
    "v2x_pubcorr",
]

# ── Variables categóricas nominales (para CatBoost) ───────────────────────────
# X_004 excluida: 627 categorías únicas, 94% códigos nuevos en SP3_test,
# codificación inconsistente entre olas, sin señal predictiva (r≈0).
# Sustituida por X_008 (tamaño ciudad, 8 categorías estables).
VARS_CATEGORICAS_NOMINALES = ["S_200", "S_700"]

# ── Variables ordinales de Likert a invertir en LB (escala 1=Mucho → 4=Nada) ──
VARS_LB_INVERTIR_LIKERT_4 = [
    "H_002_011", "H_002_031", "H_002_041", "H_002_101",
    "H_002_111", "H_002_131", "H_002_161", "H_002_241",
    "G_005_001",
]

VARS_LB_INVERTIR_LIKERT_5 = [
    "A_007_001",  # 1=Mucho interés → 4=Ninguno  (ajustado a max)
]

print("✓ Parámetros globales cargados.")
print(f"  GPU disponible : {torch.cuda.is_available()}")
print(f"  Usar GPU       : {USAR_GPU}")
print(f"  Dispositivo    : {DISPOSITIVO_TABNET}")
print(f"  Semilla        : {SEMILLA}")

## 3. Carga de datos consolidados

In [ ]:
# =============================================================================
# CELDA 3 — Carga de Latinobarómetro y V-Dem
# =============================================================================

print("Cargando Latinobarómetro...")
df_lb_raw = pd.read_csv(ARCHIVO_LB, low_memory=False, encoding="utf-8-sig")
print(f"  Latinobarómetro: {df_lb_raw.shape[0]:,} × {df_lb_raw.shape[1]} columnas")

print("Cargando V-Dem...")
df_vdem_raw = pd.read_csv(ARCHIVO_VDEM, low_memory=False, encoding="utf-8-sig")
print(f"  V-Dem         : {df_vdem_raw.shape[0]:,} × {df_vdem_raw.shape[1]} columnas")

# Vista rápida de tipos
print()
print("Muestra LB (3 filas, primeras 8 columnas):")
print(df_lb_raw.iloc[:3, :8].to_string())
print()
print("Muestra V-Dem (3 filas):")
print(df_vdem_raw.head(3).to_string())

## 4. Corrección de la variable año y construcción de la clave de fusión

`X_002` contiene el año calendario en la mayoría de las olas, pero en cinco olas
usa el número de investigación en lugar del año: `16=2011`, `17=2013`, `18=2015`,
`23=2023`, `24=2024`. Este error se corrige como **primer paso** del pipeline
antes de cualquier otra transformación.

La columna `año` resultante se usará como clave de fusión con V-Dem y como
criterio para los splits temporales.

In [ ]:
# =============================================================================
# CELDA 4 — Corrección de X_002 y derivación de columnas de año e ISO3
# =============================================================================

df_lb = df_lb_raw.copy()

# ── Paso 1: Corregir X_002 → año calendario ───────────────────────────────────
# Si la columna 'ola' existe (generada por 01_carga_datos.ipynb), extraer
# el año desde allí. Si no, corregir X_002 directamente.
if "ola" in df_lb.columns:
    df_lb[COL_AÑO] = df_lb["ola"].str.replace("LAT", "", regex=False).astype(int)
    print("✓ Año extraído desde columna 'ola'")
else:
    df_lb[COL_AÑO] = df_lb["X_002"].map(
        lambda x: MAPEO_NUMINVES_AÑO.get(int(x), int(x))
    )
    print("✓ Año derivado de X_002 con corrección de numinves")

# Verificar distribución
años_dist = df_lb[COL_AÑO].value_counts().sort_index()
print()
print("Distribución de registros por año:")
for año, n in años_dist.items():
    print(f"  {int(año)}: {n:,}")

# ── Paso 2: Derivar columna ISO3 desde pais_nombre ────────────────────────────
if COL_ISO3 not in df_lb.columns:
    df_lb[COL_ISO3] = df_lb[COL_PAIS].map(MAPEO_PAIS_ISO3)

n_sin_iso3 = df_lb[COL_ISO3].isna().sum()
print()
print(f"Registros sin ISO3 mapeado: {n_sin_iso3:,}")
if n_sin_iso3 > 0:
    print("  Países no reconocidos:", df_lb[df_lb[COL_ISO3].isna()][COL_PAIS].unique())

print()
print(f"✓ Dataset LB con año e ISO3: {df_lb.shape}")
del df_lb_raw
gc.collect()

## 5. Fusión Latinobarómetro × V-Dem

La fusión se realiza por la clave compuesta `(pais_iso3, año)`. Los indicadores
V-Dem son nivel país-año: cada encuestado individual recibe el valor institucional
del país en el año en que fue encuestado.

In [ ]:
# =============================================================================
# CELDA 5 — Merge LB × V-Dem
# =============================================================================

# Preparar V-Dem para el merge
df_vdem = df_vdem_raw.copy()

# Renombrar columnas de identificación de V-Dem para evitar conflictos
df_vdem = df_vdem.rename(columns={
    "country_text_id": COL_ISO3,
    "year"           : COL_AÑO,
})

# Excluir variables V-Dem descartadas por redundancia o ausencia en SP3_test
cols_vdem_excluir = ["country_name", "country_id", "COWcode"] + VARS_EXCLUIR_VDEM
cols_vdem_mantener = [c for c in df_vdem.columns if c not in cols_vdem_excluir]
df_vdem = df_vdem[cols_vdem_mantener]

print(f"Variables V-Dem a fusionar: {len([c for c in cols_vdem_mantener if c not in [COL_ISO3, COL_AÑO]])}")

# Realizar el merge (left join: se conservan todos los registros de LB)
df = df_lb.merge(
    df_vdem,
    on=[COL_ISO3, COL_AÑO],
    how="left",
    validate="m:1"   # Muchos registros LB por cada par (país, año) de V-Dem
)

# Verificar cobertura del merge
n_total = len(df)
n_sin_vdem = df["v2x_polyarchy"].isna().sum()
pct_merge = (1 - n_sin_vdem / n_total) * 100

print()
print(f"Resultado del merge:")
print(f"  Total registros    : {n_total:,}")
print(f"  Con V-Dem completo : {n_total - n_sin_vdem:,} ({pct_merge:.1f}%)")
print(f"  Sin V-Dem          : {n_sin_vdem:,}")
print(f"  Columnas totales   : {df.shape[1]}")

del df_lb, df_vdem, df_vdem_raw
gc.collect()

## 6. Exclusiones iniciales y tratamiento de Venezuela/Nicaragua

Se aplican las exclusiones de variables y registros definidas en la metodología
antes de cualquier transformación de escala.

In [ ]:
# =============================================================================
# CELDA 6 — Exclusiones de variables y registros especiales
# =============================================================================

# ── Paso 1: Excluir variables LB descartadas ──────────────────────────────────
cols_excluir_lb_presentes = [c for c in VARS_EXCLUIR_LB if c in df.columns]
df = df.drop(columns=cols_excluir_lb_presentes)
print(f"Variables LB excluidas: {cols_excluir_lb_presentes}")

# ── Paso 2: Excluir registros sin target válido ────────────────────────────────
# NS/NR del target (A_003_031): valores negativos o NaN
mask_target_invalido = (
    df["A_003_031"].isna() |
    df["A_003_031"].isin(NSNR_GENERICOS)
)
n_excluidos_target = mask_target_invalido.sum()
df = df[~mask_target_invalido].copy()
print(f"Registros excluidos por target NS/NR: {n_excluidos_target:,}")

# ── Paso 3: Venezuela — excluir registros post-2017 ──────────────────────────
# Venezuela se incluye en entrenamiento solo hasta 2017 inclusive.
# Sus datos post-2017 en LB tienen cuestionable representatividad.
mask_ven_post2017 = (
    (df[COL_PAIS] == "Venezuela") &
    (df[COL_AÑO] > AÑO_CORTE_VENEZUELA)
)
n_excluidos_ven = mask_ven_post2017.sum()
df = df[~mask_ven_post2017].copy()
print(f"Registros Venezuela post-{AÑO_CORTE_VENEZUELA} excluidos: {n_excluidos_ven:,}")

# ── Paso 4: Registros sin año válido ──────────────────────────────────────────
años_validos = set()
for sp_cfg in SUBPERIODOS.values():
    años_validos.update(sp_cfg["train_olas"] + sp_cfg["test_ola"])

n_antes = len(df)
df = df[df[COL_AÑO].isin(años_validos)].copy()
n_excluidos_año = n_antes - len(df)
print(f"Registros con año fuera del rango del estudio: {n_excluidos_año:,}")

print()
print(f"Dataset tras exclusiones: {df.shape[0]:,} × {df.shape[1]}")
print(f"Países: {df[COL_PAIS].nunique()}")
print(f"Años  : {sorted(df[COL_AÑO].unique())}")

gc.collect()

## 7. Recodificación de la variable dependiente

`A_003_031` se recodifica a escala base 0 `{0,1,2,3}` donde:
- `0 = Para nada satisfecho` (mínimo)
- `3 = Muy satisfecho` (máximo)

Esta recodificación es necesaria para la Regresión Logística Ordinal (`mord`)
y para XGBoost con `objective='multi:softprob'`, ambos esperan clases indexadas
desde cero.

In [ ]:
# =============================================================================
# CELDA 7 — Recodificación del target a base 0
# =============================================================================

# Verificar que la escala del target es uniforme {1,2,3,4}
vals_target = sorted(df["A_003_031"].dropna().unique())
print(f"Valores únicos en A_003_031: {[int(v) for v in vals_target]}")

assert set(int(v) for v in vals_target) == {1, 2, 3, 4}, \
    f"Escala del target inesperada: {vals_target}"

# Mapeo: 1→0, 2→1, 3→2, 4→3
RECODIFICACION_TARGET = {1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3}
df[COL_TARGET] = df["A_003_031"].map(RECODIFICACION_TARGET)

# Verificar que no quedaron NaN inesperados
n_nan_target = df[COL_TARGET].isna().sum()
assert n_nan_target == 0, f"NaN inesperados en target: {n_nan_target}"

# Distribución
print()
print("Distribución del target recodificado:")
dist = df[COL_TARGET].value_counts().sort_index()
for cls, n in dist.items():
    etiqueta = ["Para nada satisfecho","No muy satisfecho",
                "Más bien satisfecho","Muy satisfecho"][int(cls)]
    print(f"  Clase {int(cls)}: {n:>8,} ({n/len(df)*100:.1f}%) — {etiqueta}")

# Calcular pesos inversos de clase para manejo del desbalance
N_CLASES = 4
n_total_target = len(df)
PESOS_CLASE = {}
for cls in range(N_CLASES):
    n_cls = (df[COL_TARGET] == cls).sum()
    PESOS_CLASE[cls] = n_total_target / (N_CLASES * n_cls)
print()
print("Pesos inversos de clase (para manejo de desbalance):")
for cls, peso in PESOS_CLASE.items():
    print(f"  Clase {cls}: {peso:.4f}")

## 8. Transformaciones de escala

Se aplica la **Convención de Direccionalidad**: mayor valor = más favorable.
Todas las transformaciones son deterministas (no aprenden parámetros de los datos)
y se aplican antes del split temporal sin riesgo de data leakage.

In [ ]:
# =============================================================================
# CELDA 8 — Transformaciones de escala (mayor = más favorable)
# =============================================================================

def convertir_nsnr_a_nan(df: pd.DataFrame, columnas: list,
                          codigos_invalidos: list) -> pd.DataFrame:
    """
    Convierte todos los códigos de NS/NR (valores negativos y otros especiales)
    a NaN en las columnas especificadas.
    DEBE aplicarse ANTES de cualquier inversión de escala para evitar
    introducir valores ficticios (ej: 5 - 0 = 5).
    """
    df = df.copy()
    for col in columnas:
        if col not in df.columns:
            continue
        mask_invalido = df[col].isin(codigos_invalidos) | (df[col] < 0)
        n_convertidos = mask_invalido.sum()
        if n_convertidos > 0:
            df.loc[mask_invalido, col] = np.nan
    return df


# ── Paso 1: NS/NR → NaN en TODAS las columnas relevantes ─────────────────────
# Esto debe ser el PRIMER paso antes de cualquier transformación numérica.
cols_a_limpiar = [c for c in df.columns
                  if c not in [COL_TARGET, COL_AÑO, COL_PESO, "A_003_031",
                                COL_ISO3, COL_PAIS, "ola"]]

df = convertir_nsnr_a_nan(df, cols_a_limpiar, NSNR_GENERICOS)
print("✓ NS/NR convertidos a NaN en todas las variables")


# ── Paso 2: Tratamientos especiales antes de invertir ─────────────────────────

# A_007_071: valor 97 = NS/NR en olas 2020-2024; valor 0 = extrema izquierda (VÁLIDO)
df.loc[df["A_007_071"] == 97, "A_007_071"] = np.nan

# C_003_003_011: valor 5 = No aplica / No trabaja (NO es intensidad de preocupación)
df.loc[df["C_003_003_011"] == 5, "C_003_003_011"] = np.nan

# S_700 (religión): 96=Otra, 97=Ninguna/Ateo → son valores VÁLIDOS, no NS/NR
# Solo eliminar valores negativos (ya hecho arriba); 96 y 97 se mantienen como categorías

# S_701 (práctica religiosa): -3 = no aplica (sin religión), no es NS/NR clásico
# Ya fue convertido a NaN en el paso 1 (está dentro de los negativos)
print("✓ Tratamientos especiales aplicados (A_007_071=97, C_003_003_011=5)")


# ── Paso 3: Invertir escala Likert 4 puntos (1=Mucho → 4=Nada) ───────────────
# Transformación: 5 - x → convierte a 1=Nada, 4=Mucho (mayor = más favorable)
for col in VARS_LB_INVERTIR_LIKERT_4:
    if col not in df.columns:
        continue
    mask_valid = df[col].notna() & df[col].between(1, 4)
    df.loc[mask_valid, col] = 5 - df.loc[mask_valid, col]
    n_fuera = (df[col].notna() & ~df[col].between(1, 4)).sum()
    if n_fuera > 0:
        df.loc[df[col].notna() & ~df[col].between(1, 4), col] = np.nan
        print(f"  ⚠ {col}: {n_fuera} valores fuera de [1,4] convertidos a NaN tras inversión")

print(f"✓ Escala Likert 4pts invertida para {len(VARS_LB_INVERTIR_LIKERT_4)} variables")


# ── Paso 4: D_001_021, D_001_041, D_001_091 — escala comparativa ─────────────
# Período 1995-2000: escala 3 pts (1=Mejor, 2=Igual, 3=Peor)
#   → Invertir: 4 - x → 1=Peor, 3=Mejor
# Período 2001-2024: escala 5 pts (1=Mucho mejor,..., 5=Mucho peor)
#   → Invertir: 6 - x → 1=Mucho peor, 5=Mucho mejor
VARS_ECONOMICAS_COMPARATIVAS = ["D_001_021", "D_001_041", "D_001_091"]

for col in VARS_ECONOMICAS_COMPARATIVAS:
    if col not in df.columns:
        continue
    # Período 1995-2000 (escala 3 puntos)
    mask_pre2001_3pts = (df[COL_AÑO] <= 2000) & df[col].between(1, 3)
    df.loc[mask_pre2001_3pts, col] = 4 - df.loc[mask_pre2001_3pts, col]

    # Período 2001-2024 (escala 5 puntos)
    mask_post2001_5pts = (df[COL_AÑO] >= 2001) & df[col].between(1, 5)
    df.loc[mask_post2001_5pts, col] = 6 - df.loc[mask_post2001_5pts, col]

print(f"✓ Variables económicas comparativas armonizadas (D_001_021/041/091)")


# ── Paso 5: B_006_061 — Aprobación gobierno binaria ──────────────────────────
# Original: 1=Aprueba, 2=Desaprueba
# Recodificar: 1→1 (Aprueba), 2→0 (Desaprueba)
if "B_006_061" in df.columns:
    df["B_006_061"] = df["B_006_061"].map({1.0: 1, 2.0: 0})
print("✓ B_006_061 recodificada: {1=Aprueba, 0=Desaprueba}")


# ── Paso 6: B_001_101 — País para todos / para poderosos ──────────────────────
# Original: 1=Para todos, 2=Para los poderosos
# Recodificar: 1→1, 2→0
if "B_001_101" in df.columns:
    df["B_001_101"] = df["B_001_101"].map({1.0: 1, 2.0: 0})
print("✓ B_001_101 recodificada: {1=Para todos, 0=Para poderosos}")


# ── Paso 7: H_001_011 — Confianza interpersonal ────────────────────────────────
# Original: 1=Se puede confiar, 2=Hay que tener cuidado
# Recodificar: 1→1, 2→0
if "H_001_011" in df.columns:
    df["H_001_011"] = df["H_001_011"].map({1.0: 1, 2.0: 0})
print("✓ H_001_011 recodificada: {1=Confianza, 0=Desconfianza}")


# ── Paso 8: I_001_001 — Victimización ─────────────────────────────────────────
# 1995-2008: 1=Sí víctima, 2=No víctima
# 2009    : 1=Entrevistado, 2=Familiar, 3=No
# 2010-2024: 1=Entrevistado, 2=Familiar, 3=Ambos, 4=No
# Recodificar a binaria: 1=Víctima (cualquier modalidad), 0=No víctima
if "I_001_001" in df.columns:
    col = "I_001_001"
    nueva_col = np.full(len(df), np.nan)

    # Período 1995-2008 (binaria original)
    mask_bin = df[COL_AÑO] <= 2008
    nueva_col[mask_bin & (df[col] == 1)] = 1
    nueva_col[mask_bin & (df[col] == 2)] = 0

    # 2009 (ternaria)
    mask_2009 = df[COL_AÑO] == 2009
    nueva_col[mask_2009 & df[col].isin([1, 2])] = 1
    nueva_col[mask_2009 & (df[col] == 3)] = 0

    # 2010-2024 (cuaternaria)
    mask_post2010 = df[COL_AÑO] >= 2010
    nueva_col[mask_post2010 & df[col].isin([1, 2, 3])] = 1
    nueva_col[mask_post2010 & (df[col] == 4)] = 0

    df[col] = nueva_col
print("✓ I_001_001 recodificada a binaria {1=Víctima, 0=No víctima}")


# ── Paso 9: G_002_011 — Conocimiento de corrupción ────────────────────────────
# La mayoría de olas: 1=Sí, 2=No
# LAT2013: escala anómala {1,2,3,4,5} — recodificar 1→1, resto→0
if "G_002_011" in df.columns:
    col = "G_002_011"
    # Olas binarias estándar
    mask_std = df[COL_AÑO] != 2013
    df.loc[mask_std & (df[col] == 1), col] = 1
    df.loc[mask_std & (df[col] == 2), col] = 0
    # LAT2013: cualquier código > 1 = No conoce directamente
    mask_2013 = df[COL_AÑO] == 2013
    df.loc[mask_2013 & (df[col] > 1), col] = 0
    df.loc[mask_2013 & (df[col] == 1), col] = 1
print("✓ G_002_011 recodificada a binaria {1=Conoce corrupción, 0=No conoce}")


# ── Paso 10: S_001 — Sexo ─────────────────────────────────────────────────────
# Original: 1=Hombre, 2=Mujer → Recodificar: 0=Hombre, 1=Mujer
if "S_001" in df.columns:
    df["S_001"] = df["S_001"].map({1.0: 0, 2.0: 1})
print("✓ S_001 recodificado: {0=Hombre, 1=Mujer}")


# ── Paso 11: Invertir índices V-Dem con dirección negativa ────────────────────
# v2x_corr, v2x_execorr, v2x_pubcorr: mayor = más corrupción → invertir 1-x
for col in VARS_VDEM_INVERTIR:
    if col in df.columns:
        df[col] = 1.0 - df[col]
print(f"✓ Índices V-Dem invertidos: {VARS_VDEM_INVERTIR}")
print("  Semántica post-inversión: mayor valor = menor corrupción")


print()
print("=" * 60)
print("Resumen de transformaciones completadas")
print(f"  Dataset final: {df.shape[0]:,} × {df.shape[1]}")
print(f"  Missingness global: {df.isnull().mean().mean()*100:.1f}%")

## 9. Definición del conjunto de features por modelo

Se definen tres conjuntos de features según el modelo:
- **OLO**: 5 high-level indices V-Dem + variables independientes seleccionadas
- **Árboles/TabNet**: mid-level indices V-Dem + todas las variables LB

In [ ]:
# =============================================================================
# CELDA 9 — Definición de features por tipo de modelo
# =============================================================================

# ── Features V-Dem high-level (solo OLO) ─────────────────────────────────────
VDEM_HIGH_LEVEL = [
    "v2x_polyarchy", "v2x_libdem", "v2x_partipdem",
    "v2x_delibdem",  "v2x_egaldem",
]

# ── Features V-Dem mid-level (árboles y TabNet) ───────────────────────────────
VDEM_MID_LEVEL = [
    # Componentes liberales
    "v2xcl_rol", "v2x_jucon", "v2xlg_legcon", "v2x_freexp_altinf",
    # Poliarquía y sociedad civil
    "v2x_cspart", "v2xcs_ccsi",
    # Igualdad y distribución
    "v2x_egal", "v2xeg_eqdr",
    # Accountability y estado de derecho
    "v2x_accountability_osp", "v2x_rule",
    # Corrupción (ya invertidos en paso 11)
    "v2x_corr", "v2x_execorr", "v2x_pubcorr",
    # Género
    "v2x_gender",
]

# ── Features Latinobarómetro ──────────────────────────────────────────────────
LB_FEATURES = [
    # Apoyo democrático
    "A_001_001",
    # Confianza institucional (ya invertidas)
    "H_002_011", "H_002_031", "H_002_041", "H_002_101",
    "H_002_111", "H_002_131", "H_002_161", "H_002_241",
    "H_001_011",
    # Evaluación económica
    "D_001_001", "D_001_041", "D_001_091",
    # Comparativas (ya armonizadas)
    "D_001_021",
    # Distribución y trabajo
    "C_003_003_011", "C_006_003_011",
    # Percepción política
    "B_001_101", "B_006_061",
    "A_007_071", "A_007_001",
    # Corrupción y seguridad
    "G_002_011", "G_005_001", "I_001_001",
    # Sociodemográfico
    "S_001", "S_002", "S_101", "S_200", "S_301", "S_700", "S_701",
    # Urbano-rural
    "X_008",
]

# Filtrar solo columnas que existen en el dataframe
LB_FEATURES         = [c for c in LB_FEATURES         if c in df.columns]
VDEM_HIGH_LEVEL     = [c for c in VDEM_HIGH_LEVEL     if c in df.columns]
VDEM_MID_LEVEL      = [c for c in VDEM_MID_LEVEL      if c in df.columns]
VARS_CATEGORICAS    = [c for c in VARS_CATEGORICAS_NOMINALES if c in df.columns]

# Conjuntos finales por tipo de modelo
FEATURES_OLO    = LB_FEATURES + VDEM_HIGH_LEVEL
FEATURES_TREES  = LB_FEATURES + VDEM_MID_LEVEL  # XGBoost, CatBoost, LightGBM, TabNet

print(f"Features OLO              : {len(FEATURES_OLO)}")
print(f"Features Árboles/TabNet   : {len(FEATURES_TREES)}")
print(f"Variables categóricas     : {VARS_CATEGORICAS}")
print()
print("Distribución de features:")
print(f"  LB features             : {len(LB_FEATURES)}")
print(f"  V-Dem high-level (OLO)  : {len(VDEM_HIGH_LEVEL)}")
print(f"  V-Dem mid-level (trees) : {len(VDEM_MID_LEVEL)}")

## 10. Construcción de los splits temporales (Expanding Window)

Función que genera los tres conjuntos train/test siguiendo el esquema
*Walk-Forward Validation* con ventana expandiente.

In [ ]:
# =============================================================================
# CELDA 10 — Función de construcción de splits temporales
# =============================================================================

def construir_split(
    df: pd.DataFrame,
    subperiodo: str,
    features: list,
    col_target: str = COL_TARGET,
    col_año: str = COL_AÑO,
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series,
           pd.Series, pd.Series]:
    """
    Construye el par (train, test) para un subperiodo dado siguiendo
    el esquema Expanding Window Walk-Forward Validation.

    Parámetros
    ----------
    df         : DataFrame consolidado tras preprocesamiento.
    subperiodo : Clave del subperiodo ('SP1', 'SP2' o 'SP3').
    features   : Lista de columnas a usar como features.
    col_target : Nombre de la columna target.
    col_año    : Nombre de la columna de año.

    Retorna
    -------
    X_train, y_train, X_test, y_test, w_train, w_test
        Donde w son los sample_weights (factor de expansión muestral
        normalizado × peso inverso de clase).
    """
    cfg         = SUBPERIODOS[subperiodo]
    años_train  = cfg["train_olas"]
    años_test   = cfg["test_ola"]

    # Filtrar por años
    df_train = df[df[col_año].isin(años_train)].copy()
    df_test  = df[df[col_año].isin(años_test)].copy()

    # Filtrar solo features existentes
    feats_disponibles = [f for f in features if f in df.columns]

    X_train = df_train[feats_disponibles]
    y_train = df_train[col_target].astype(int)
    X_test  = df_test[feats_disponibles]
    y_test  = df_test[col_target].astype(int)

    # ── Calcular sample_weight combinado ─────────────────────────────────────
    # Componente 1: factor de expansión muestral (X_020 normalizado)
    w_muestral_train = df_train[COL_PESO].fillna(1.0)
    w_muestral_train = w_muestral_train / w_muestral_train.mean()

    w_muestral_test  = df_test[COL_PESO].fillna(1.0)
    w_muestral_test  = w_muestral_test / w_muestral_test.mean()

    # Componente 2: peso inverso de clase
    w_clase_train = y_train.map(PESOS_CLASE)
    w_clase_test  = y_test.map(PESOS_CLASE)

    # Peso final combinado
    w_train = (w_muestral_train.values * w_clase_train.values).astype(float)
    w_test  = (w_muestral_test.values  * w_clase_test.values).astype(float)

    return X_train, y_train, X_test, y_test, w_train, w_test


def resumen_split(nombre: str, X_train, y_train, X_test, y_test):
    """Imprime un resumen diagnóstico del split."""
    print(f"\n{'─'*55}")
    print(f"  {nombre}")
    print(f"{'─'*55}")
    print(f"  Train: {len(X_train):>8,} registros | {X_train.shape[1]} features")
    print(f"  Test : {len(X_test):>8,} registros")
    print(f"  Ratio train/test: {len(X_train)/len(X_test):.1f}x")
    print(f"  Clases en train: ", dict(y_train.value_counts().sort_index()))
    print(f"  Clases en test : ", dict(y_test.value_counts().sort_index()))
    pct_nan_train = X_train.isnull().mean().mean() * 100
    pct_nan_test  = X_test.isnull().mean().mean() * 100
    print(f"  Missingness train: {pct_nan_train:.1f}% | test: {pct_nan_test:.1f}%")


print("✓ Funciones de split definidas.")

## 11. Imputación diferenciada por modelo

Se implementan dos estrategias:
- **OLO y TabNet**: `IterativeImputer` con `BayesianRidge` (MICE). Fit exclusivo sobre train, transform sobre train y test.
- **XGBoost, LightGBM, CatBoost**: datos con NaN nativos (los tres modelos manejan valores faltantes internamente).

> **Regla crítica anti-data leakage**: el imputador se ajusta **exclusivamente** sobre el conjunto de entrenamiento de cada split.

In [ ]:
# =============================================================================
# CELDA 11 — Funciones de imputación diferenciada
# =============================================================================

def imputar_para_olo_tabnet(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    semilla: int = SEMILLA,
) -> Tuple[pd.DataFrame, pd.DataFrame, IterativeImputer]:
    """
    Aplica imputación MICE (IterativeImputer con BayesianRidge) para los
    modelos OLO y TabNet que requieren matrices completas.

    El imputador se ajusta SOLO sobre X_train y se transforma sobre ambos.
    Las columnas categóricas nominales se excluyen de la imputación y se
    tratan por separado con SimpleImputer (moda).

    Retorna: X_train_imp, X_test_imp, imputer_ajustado
    """
    # Separar columnas: continuas/ordinales vs categóricas nominales
    cols_categoricas = [c for c in VARS_CATEGORICAS if c in X_train.columns]
    cols_numericas   = [c for c in X_train.columns if c not in cols_categoricas]

    # Imputación MICE para columnas numéricas
    imputer_mice = IterativeImputer(
        estimator    = BayesianRidge(),
        max_iter     = 10,
        random_state = semilla,
        verbose      = 0,
    )
    X_train_num = pd.DataFrame(
        imputer_mice.fit_transform(X_train[cols_numericas]),
        columns = cols_numericas,
        index   = X_train.index,
    )
    X_test_num  = pd.DataFrame(
        imputer_mice.transform(X_test[cols_numericas]),
        columns = cols_numericas,
        index   = X_test.index,
    )

    # Imputación por moda para columnas categóricas
    if cols_categoricas:
        imputer_cat = SimpleImputer(strategy="most_frequent")
        X_train_cat = pd.DataFrame(
            imputer_cat.fit_transform(X_train[cols_categoricas]),
            columns = cols_categoricas,
            index   = X_train.index,
        )
        X_test_cat = pd.DataFrame(
            imputer_cat.transform(X_test[cols_categoricas]),
            columns = cols_categoricas,
            index   = X_test.index,
        )
        X_train_imp = pd.concat([X_train_num, X_train_cat], axis=1)
        X_test_imp  = pd.concat([X_test_num,  X_test_cat],  axis=1)
    else:
        X_train_imp = X_train_num
        X_test_imp  = X_test_num

    # Reordenar columnas al orden original
    X_train_imp = X_train_imp[X_train.columns]
    X_test_imp  = X_test_imp[X_test.columns]

    return X_train_imp, X_test_imp, imputer_mice


def normalizar(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    metodo: str = "minmax",
) -> Tuple[pd.DataFrame, pd.DataFrame, object]:
    """
    Normaliza features continuas. Fit en train, transform en ambos.

    metodo : 'minmax' para TabNet | 'standard' para OLO
    """
    cols_num = [c for c in X_train.columns if c not in VARS_CATEGORICAS]

    scaler = MinMaxScaler() if metodo == "minmax" else StandardScaler()

    X_train_sc = X_train.copy()
    X_test_sc  = X_test.copy()

    X_train_sc[cols_num] = scaler.fit_transform(X_train[cols_num])
    X_test_sc[cols_num]  = scaler.transform(X_test[cols_num])

    return X_train_sc, X_test_sc, scaler


print("✓ Funciones de imputación y normalización definidas.")

## 12. Función de evaluación unificada

In [ ]:
# =============================================================================
# CELDA 12 — Función de evaluación unificada para todos los modelos
# =============================================================================

def evaluar_modelo(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: Optional[np.ndarray] = None,
    nombre: str = "",
    subperiodo: str = "",
) -> Dict:
    """
    Calcula el conjunto completo de métricas para clasificación ordinal
    multiclase.

    Métricas reportadas:
    - Balanced Accuracy
    - F1 Macro
    - F1 Weighted
    - Cohen's Kappa (lineal y cuadrático — apropiado para escalas ordinales)
    - MAE Ordinal (error absoluto medio sobre clases {0,1,2,3})
    - AUROC (OvR macro, si hay probabilidades disponibles)
    """
    metricas = {
        "modelo"              : nombre,
        "subperiodo"          : subperiodo,
        "balanced_accuracy"   : balanced_accuracy_score(y_true, y_pred),
        "f1_macro"            : f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "f1_weighted"         : f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa_lineal"        : cohen_kappa_score(y_true, y_pred, weights="linear"),
        "kappa_cuadratico"    : cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "mae_ordinal"         : mean_absolute_error(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            metricas["auroc_macro"] = roc_auc_score(
                y_true, y_prob, multi_class="ovr", average="macro"
            )
        except Exception:
            metricas["auroc_macro"] = np.nan
    else:
        metricas["auroc_macro"] = np.nan

    if nombre:
        print(f"  {'─'*50}")
        print(f"  {nombre} — {subperiodo}")
        print(f"  {'─'*50}")
        for k, v in metricas.items():
            if k in ["modelo", "subperiodo"]:
                continue
            print(f"    {k:<25}: {v:.4f}" if isinstance(v, float) else f"    {k}: {v}")

    return metricas


# Almacenamiento global de resultados
RESULTADOS_GLOBALES: List[Dict] = []

print("✓ Función de evaluación definida.")

## 13. Modelo 1 — Regresión Logística Ordinal (Baseline)

La Regresión Logística Ordinal actúa como **línea base estadística interpretable**.
Se usa la librería `mord` (`LogisticIT`: Immediate Threshold — threshold model con
logit link). La optimización del hiperparámetro `alpha` (regularización L2)
se realiza mediante Optuna con `TimeSeriesSplit` interno sobre el train.

In [ ]:
# =============================================================================
# CELDA 13 — Regresión Logística Ordinal (mord.LogisticIT)
# =============================================================================
try:
    import mord
    MORD_DISPONIBLE = True
    print("✓ mord disponible")
except ImportError:
    print("⚠ mord no instalado. Instalar con: pip install mord")
    print("  Usando LogisticRegression multiclase como fallback.")
    MORD_DISPONIBLE = False
    from sklearn.linear_model import LogisticRegression


def entrenar_olo(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    w_train: np.ndarray,
    subperiodo: str,
    hp_guardados: Optional[Dict] = None,
) -> Dict:
    """
    Entrena la Regresión Logística Ordinal para un subperiodo dado.

    La búsqueda de hiperparámetros optimiza el Kappa cuadrático en
    validación temporal (TimeSeriesSplit). Si hp_guardados se provee
    y EJECUTAR_BUSQUEDA_HP=False, se usan directamente.
    """
    nombre_modelo = "OLO"
    print(f"\n{'='*55}")
    print(f"  Entrenando {nombre_modelo} — {subperiodo}")
    print(f"{'='*55}")

    # ── Búsqueda de hiperparámetros ───────────────────────────────────────────
    ruta_hp = CARPETA_MODELS / f"hp_{nombre_modelo}_{subperiodo}.json"

    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        with open(ruta_hp) as f:
            best_hp = json.load(f)
        print(f"  Hiperparámetros cargados desde: {ruta_hp.name}")
    else:
        print(f"  Ejecutando búsqueda con Optuna ({N_TRIALS_OPTUNA} trials)...")
        tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)

        def objetivo(trial):
            alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
            scores = []
            for idx_tr, idx_val in tscv.split(X_train):
                Xtr, Xval = X_train[idx_tr], X_train[idx_val]
                ytr, yval = y_train[idx_tr], y_train[idx_val]
                wtr       = w_train[idx_tr]
                if MORD_DISPONIBLE:
                    clf = mord.LogisticIT(alpha=alpha, max_iter=500)
                    clf.fit(Xtr, ytr, sample_weight=wtr)
                else:
                    clf = LogisticRegression(
                        C=1/alpha,
                        max_iter=500, random_state=SEMILLA
                    )
                    clf.fit(Xtr, ytr, sample_weight=wtr)
                y_val_pred = clf.predict(Xval)
                scores.append(
                    cohen_kappa_score(yval, y_val_pred, weights="quadratic")
                )
            return np.mean(scores)

        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=SEMILLA),
        )
        study.optimize(objetivo, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa CV: {study.best_value:.4f} | Params: {best_hp}")
        with open(ruta_hp, "w") as f:
            json.dump(best_hp, f)

    # ── Entrenamiento final ────────────────────────────────────────────────────
    alpha_opt = best_hp.get("alpha", 1.0)
    if MORD_DISPONIBLE:
        clf_final = mord.LogisticIT(alpha=alpha_opt, max_iter=500)
        clf_final.fit(X_train, y_train, sample_weight=w_train)
        y_pred = clf_final.predict(X_test)
        y_prob = clf_final.predict_proba(X_test)
    else:
        clf_final = LogisticRegression(
            C=1/alpha_opt,
            max_iter=500, random_state=SEMILLA
        )
        clf_final.fit(X_train, y_train, sample_weight=w_train)
        y_pred = clf_final.predict(X_test)
        y_prob = clf_final.predict_proba(X_test)

    # Guardar modelo
    ruta_modelo = CARPETA_MODELS / f"{nombre_modelo}_{subperiodo}.pkl"
    joblib.dump(clf_final, ruta_modelo)
    print(f"  Modelo guardado: {ruta_modelo.name}")

    # Evaluar
    metricas = evaluar_modelo(y_test, y_pred, y_prob, nombre_modelo, subperiodo)
    RESULTADOS_GLOBALES.append(metricas)
    return metricas


print("✓ Función de entrenamiento OLO definida.")

## 14. Modelo 2 — XGBoost

In [ ]:
# =============================================================================
# CELDA 14 — XGBoost con Optuna
# =============================================================================

def entrenar_xgboost(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    w_train: np.ndarray,
    subperiodo: str,
    hp_guardados: Optional[Dict] = None,
) -> Dict:
    """
    Entrena XGBoost para clasificación ordinal multiclase.
    Usa sample_weight para manejo de desbalance (sin SMOTE).
    XGBoost maneja NaN nativamente — no requiere imputación previa.
    """
    nombre_modelo = "XGBoost"
    print(f"\n{'='*55}")
    print(f"  Entrenando {nombre_modelo} — {subperiodo}")
    print(f"{'='*55}")

    ruta_hp = CARPETA_MODELS / f"hp_{nombre_modelo}_{subperiodo}.json"

    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        with open(ruta_hp) as f:
            best_hp = json.load(f)
        print(f"  Hiperparámetros cargados desde: {ruta_hp.name}")
    else:
        print(f"  Ejecutando búsqueda con Optuna ({N_TRIALS_OPTUNA} trials)...")
        tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)

        def objetivo(trial):
            params = {
                "n_estimators"      : trial.suggest_int("n_estimators", 200, 1000, step=100),
                "max_depth"         : trial.suggest_int("max_depth", 3, 8),
                "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample"         : trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "min_child_weight"  : trial.suggest_int("min_child_weight", 1, 10),
                "reg_alpha"         : trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda"        : trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "objective"         : "multi:softprob",
                "num_class"         : N_CLASES,
                "eval_metric"       : "mlogloss",
                "tree_method"       : "hist",
                "device"            : "cuda" if USAR_GPU else "cpu",
                "random_state"      : SEMILLA,
                "n_jobs"            : N_JOBS,
                "verbosity"         : 0,
            }
            scores = []
            for idx_tr, idx_val in tscv.split(X_train):
                Xtr  = X_train.iloc[idx_tr]
                Xval = X_train.iloc[idx_val]
                ytr  = y_train.iloc[idx_tr]
                yval = y_train.iloc[idx_val]
                wtr  = w_train[idx_tr]
                clf  = xgb.XGBClassifier(**params)
                clf.fit(Xtr, ytr, sample_weight=wtr,
                        eval_set=[(Xval, yval)],
                        verbose=False)
                y_val_pred = clf.predict(Xval)
                scores.append(
                    cohen_kappa_score(yval, y_val_pred, weights="quadratic")
                )
            return np.mean(scores)

        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=SEMILLA),
        )
        study.optimize(objetivo, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa CV: {study.best_value:.4f} | Params: {best_hp}")
        with open(ruta_hp, "w") as f:
            json.dump(best_hp, f)

    # ── Entrenamiento final ────────────────────────────────────────────────────
    params_final = {
        **best_hp,
        "objective"  : "multi:softprob",
        "num_class"  : N_CLASES,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "device"     : "cuda" if USAR_GPU else "cpu",
        "random_state": SEMILLA,
        "n_jobs"     : N_JOBS,
        "verbosity"  : 0,
    }
    clf_final = xgb.XGBClassifier(**params_final)
    clf_final.fit(X_train, y_train, sample_weight=w_train, verbose=False)

    y_pred = clf_final.predict(X_test)
    y_prob = clf_final.predict_proba(X_test)

    ruta_modelo = CARPETA_MODELS / f"{nombre_modelo}_{subperiodo}.pkl"
    joblib.dump(clf_final, ruta_modelo)
    print(f"  Modelo guardado: {ruta_modelo.name}")

    metricas = evaluar_modelo(y_test, y_pred, y_prob, nombre_modelo, subperiodo)
    RESULTADOS_GLOBALES.append(metricas)
    return metricas


print("✓ Función de entrenamiento XGBoost definida.")

## 15. Modelo 3 — CatBoost

In [ ]:
# =============================================================================
# CELDA 15 — CatBoost con Optuna
# CatBoost recibe variables categóricas nativas via cat_features.
# No requiere imputación previa (maneja NaN internamente).
# =============================================================================

def entrenar_catboost(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    w_train: np.ndarray,
    subperiodo: str,
) -> Dict:
    """
    Entrena CatBoost con soporte nativo para variables categóricas nominales.
    Las columnas en VARS_CATEGORICAS se pasan como cat_features.
    No requiere imputación previa ni OrdinalEncoder.
    """
    nombre_modelo = "CatBoost"
    print(f"\n{'='*55}")
    print(f"  Entrenando {nombre_modelo} — {subperiodo}")
    print(f"{'='*55}")

    # Índices de columnas categóricas
    cat_feature_indices = [
        i for i, c in enumerate(X_train.columns)
        if c in VARS_CATEGORICAS
    ]

    # Convertir NaN en categóricas a string 'None' (CatBoost requiere esto)
    X_train_cat = X_train.copy()
    X_test_cat  = X_test.copy()
    for col in VARS_CATEGORICAS:
        if col in X_train_cat.columns:
            X_train_cat[col] = X_train_cat[col].fillna(-999).astype(int).astype(str)
            X_test_cat[col]  = X_test_cat[col].fillna(-999).astype(int).astype(str)

    ruta_hp = CARPETA_MODELS / f"hp_{nombre_modelo}_{subperiodo}.json"

    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        with open(ruta_hp) as f:
            best_hp = json.load(f)
        print(f"  Hiperparámetros cargados desde: {ruta_hp.name}")
    else:
        print(f"  Ejecutando búsqueda con Optuna ({N_TRIALS_OPTUNA} trials)...")
        tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)

        def objetivo(trial):
            params = {
                "iterations"         : trial.suggest_int("iterations", 300, 1000, step=100),
                "depth"              : trial.suggest_int("depth", 4, 8),
                "learning_rate"      : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "l2_leaf_reg"        : trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "border_count"       : trial.suggest_int("border_count", 32, 128),
                "random_strength"    : trial.suggest_float("random_strength", 0.0, 10.0),
            }
            scores = []
            for idx_tr, idx_val in tscv.split(X_train_cat):
                Xtr  = X_train_cat.iloc[idx_tr]
                Xval = X_train_cat.iloc[idx_val]
                ytr  = y_train.iloc[idx_tr].values
                yval = y_train.iloc[idx_val].values
                wtr  = w_train[idx_tr]
                pool_tr  = Pool(Xtr, label=ytr, weight=wtr,
                               cat_features=cat_feature_indices)
                pool_val = Pool(Xval, label=yval,
                               cat_features=cat_feature_indices)
                clf = CatBoostClassifier(
                    **params,
                    loss_function="MultiClass",
                    random_seed=SEMILLA,
                    verbose=False,
                    task_type="GPU" if USAR_GPU else "CPU",
                )
                clf.fit(pool_tr, eval_set=pool_val)
                y_val_pred = clf.predict(Xval).flatten()
                scores.append(
                    cohen_kappa_score(yval, y_val_pred, weights="quadratic")
                )
            return np.mean(scores)

        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=SEMILLA),
        )
        study.optimize(objetivo, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa CV: {study.best_value:.4f} | Params: {best_hp}")
        with open(ruta_hp, "w") as f:
            json.dump(best_hp, f)

    # ── Entrenamiento final ────────────────────────────────────────────────────
    pool_train_final = Pool(
        X_train_cat, label=y_train.values, weight=w_train,
        cat_features=cat_feature_indices,
    )
    clf_final = CatBoostClassifier(
        **best_hp,
        loss_function="MultiClass",
        random_seed=SEMILLA,
        verbose=False,
        task_type="GPU" if USAR_GPU else "CPU",
    )
    clf_final.fit(pool_train_final)

    y_pred = clf_final.predict(X_test_cat).flatten()
    y_prob = clf_final.predict_proba(X_test_cat)

    ruta_modelo = CARPETA_MODELS / f"{nombre_modelo}_{subperiodo}.cbm"
    clf_final.save_model(str(ruta_modelo))
    print(f"  Modelo guardado: {ruta_modelo.name}")

    metricas = evaluar_modelo(y_test, y_pred, y_prob, nombre_modelo, subperiodo)
    RESULTADOS_GLOBALES.append(metricas)
    return metricas


print("✓ Función de entrenamiento CatBoost definida.")

## 16. Modelo 4 — LightGBM

In [ ]:
# =============================================================================
# CELDA 16 — LightGBM con Optuna
# LightGBM maneja NaN nativamente y soporta variables categóricas vía dtype.
# =============================================================================

def entrenar_lightgbm(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    w_train: np.ndarray,
    subperiodo: str,
) -> Dict:
    """
    Entrena LightGBM para clasificación multiclase.
    Las variables categóricas se codifican con OrdinalEncoder (LightGBM
    acepta dtype 'category' de pandas directamente).
    No requiere imputación previa.
    """
    nombre_modelo = "LightGBM"
    print(f"\n{'='*55}")
    print(f"  Entrenando {nombre_modelo} — {subperiodo}")
    print(f"{'='*55}")

    # Convertir categóricas a dtype 'category' de pandas
    X_train_lgb = X_train.copy()
    X_test_lgb  = X_test.copy()
    for col in VARS_CATEGORICAS:
        if col in X_train_lgb.columns:
            # Asegurar que train y test compartan el mismo universo de categorías
            cats = sorted(set(
                X_train_lgb[col].dropna().unique().tolist() +
                X_test_lgb[col].dropna().unique().tolist()
            ))
            cat_type = pd.CategoricalDtype(categories=cats, ordered=False)
            X_train_lgb[col] = X_train_lgb[col].astype(cat_type)
            X_test_lgb[col]  = X_test_lgb[col].astype(cat_type)

    ruta_hp = CARPETA_MODELS / f"hp_{nombre_modelo}_{subperiodo}.json"

    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        with open(ruta_hp) as f:
            best_hp = json.load(f)
        print(f"  Hiperparámetros cargados desde: {ruta_hp.name}")
    else:
        print(f"  Ejecutando búsqueda con Optuna ({N_TRIALS_OPTUNA} trials)...")
        tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)

        def objetivo(trial):
            params = {
                "n_estimators"      : trial.suggest_int("n_estimators", 200, 1000, step=100),
                "num_leaves"        : trial.suggest_int("num_leaves", 20, 150),
                "max_depth"         : trial.suggest_int("max_depth", 3, 8),
                "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample"         : trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha"         : trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda"        : trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "min_child_samples" : trial.suggest_int("min_child_samples", 5, 100),
            }
            scores = []
            for idx_tr, idx_val in tscv.split(X_train_lgb):
                Xtr  = X_train_lgb.iloc[idx_tr]
                Xval = X_train_lgb.iloc[idx_val]
                ytr  = y_train.iloc[idx_tr]
                yval = y_train.iloc[idx_val]
                wtr  = w_train[idx_tr]
                clf = lgb.LGBMClassifier(
                    **params,
                    objective="multiclass",
                    num_class=N_CLASES,
                    class_weight=PESOS_CLASE,
                    random_state=SEMILLA,
                    n_jobs=N_JOBS,
                    verbose=-1,
                    device="gpu" if USAR_GPU else "cpu",
                )
                clf.fit(
                    Xtr, ytr, sample_weight=wtr,
                    eval_set=[(Xval, yval)],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(-1)],
                )
                y_val_pred = clf.predict(Xval)
                scores.append(
                    cohen_kappa_score(yval, y_val_pred, weights="quadratic")
                )
            return np.mean(scores)

        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=SEMILLA),
        )
        study.optimize(objetivo, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa CV: {study.best_value:.4f} | Params: {best_hp}")
        with open(ruta_hp, "w") as f:
            json.dump(best_hp, f)

    # ── Entrenamiento final ────────────────────────────────────────────────────
    clf_final = lgb.LGBMClassifier(
        **best_hp,
        objective="multiclass",
        num_class=N_CLASES,
        class_weight=PESOS_CLASE,
        random_state=SEMILLA,
        n_jobs=N_JOBS,
        verbose=-1,
        device="gpu" if USAR_GPU else "cpu",
    )
    clf_final.fit(X_train_lgb, y_train, sample_weight=w_train,
                  callbacks=[lgb.log_evaluation(-1)])

    y_pred = clf_final.predict(X_test_lgb)
    y_prob = clf_final.predict_proba(X_test_lgb)

    ruta_modelo = CARPETA_MODELS / f"{nombre_modelo}_{subperiodo}.pkl"
    joblib.dump(clf_final, ruta_modelo)
    print(f"  Modelo guardado: {ruta_modelo.name}")

    metricas = evaluar_modelo(y_test, y_pred, y_prob, nombre_modelo, subperiodo)
    RESULTADOS_GLOBALES.append(metricas)
    return metricas


print("✓ Función de entrenamiento LightGBM definida.")

## 17. Modelo 5 — TabNet

In [ ]:
# =============================================================================
# CELDA 17 — TabNet con Optuna
# TabNet requiere: matrices completas (sin NaN) y MinMaxScaler [0,1].
# Las variables categóricas se pasan como embeddings de dimensión reducida.
# NOTA: TabNet no soporta sample_weight individual — se documenta limitación.
# =============================================================================

def entrenar_tabnet(
    X_train_imp: np.ndarray,
    y_train: np.ndarray,
    X_test_imp: np.ndarray,
    y_test: np.ndarray,
    subperiodo: str,
    cat_idxs: List[int],
    cat_dims: List[int],
) -> Dict:
    """
    Entrena TabNet para clasificación ordinal multiclase.

    Limitación documentada: pytorch-tabnet no soporta sample_weight
    por observación individual. Se usan pesos por clase (weights)
    como alternativa. El análisis de sensibilidad por país en el test
    compensa esta limitación (ver análisis de estabilidad regional).

    Parámetros
    ----------
    X_train_imp : Array imputado y normalizado con MinMaxScaler.
    cat_idxs    : Índices de columnas categóricas para embeddings.
    cat_dims    : Cardinalidad de cada columna categórica.
    """
    nombre_modelo = "TabNet"
    print(f"\n{'='*55}")
    print(f"  Entrenando {nombre_modelo} — {subperiodo}")
    print(f"  Dispositivo: {DISPOSITIVO_TABNET}")
    print(f"{'='*55}")

    # Pesos por clase para TabNet (alternativa a sample_weight)
    class_weights_array = np.array([PESOS_CLASE[i] for i in range(N_CLASES)])

    ruta_hp = CARPETA_MODELS / f"hp_{nombre_modelo}_{subperiodo}.json"

    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        with open(ruta_hp) as f:
            best_hp = json.load(f)
        print(f"  Hiperparámetros cargados desde: {ruta_hp.name}")
    else:
        print(f"  Ejecutando búsqueda con Optuna ({N_TRIALS_OPTUNA} trials)...")
        tscv = TimeSeriesSplit(n_splits=min(N_SPLITS_CV, 3))  # 3 folds para TabNet (más lento)

        def objetivo(trial):
            params = {
                "n_d"              : trial.suggest_int("n_d", 8, 64, step=8),
                "n_a"              : trial.suggest_int("n_a", 8, 64, step=8),
                "n_steps"          : trial.suggest_int("n_steps", 3, 7),
                "gamma"            : trial.suggest_float("gamma", 1.0, 2.0),
                "lambda_sparse"    : trial.suggest_float("lambda_sparse", 1e-6, 1e-3, log=True),
                "momentum"         : trial.suggest_float("momentum", 0.01, 0.4),
                "mask_type"        : trial.suggest_categorical("mask_type", ["sparsemax", "entmax"]),
            }
            lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
            scores = []
            for idx_tr, idx_val in tscv.split(X_train_imp):
                Xtr  = X_train_imp[idx_tr].astype(np.float32)
                Xval = X_train_imp[idx_val].astype(np.float32)
                ytr  = y_train[idx_tr]
                yval = y_train[idx_val]
                clf = TabNetClassifier(
                    **params,
                    optimizer_fn=torch.optim.Adam,
                    optimizer_params={"lr": lr},
                    scheduler_fn=torch.optim.lr_scheduler.StepLR,
                    scheduler_params={"step_size": 10, "gamma": 0.9},
                    cat_idxs=cat_idxs,
                    cat_dims=cat_dims,
                    cat_emb_dim=3,
                    verbose=0,
                    device_name=DISPOSITIVO_TABNET,
                    seed=SEMILLA,
                )
                clf.fit(
                    Xtr, ytr,
                    eval_set=[(Xval, yval)],
                    eval_metric=["balanced_accuracy"],
                    max_epochs=100,
                    patience=15,
                    batch_size=1024,
                    virtual_batch_size=128,
                    weights=1,  # pesos inversos automáticos por clase
                )
                y_val_pred = clf.predict(Xval)
                scores.append(
                    cohen_kappa_score(yval, y_val_pred, weights="quadratic")
                )
            return np.mean(scores)

        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=SEMILLA),
        )
        study.optimize(objetivo, n_trials=min(N_TRIALS_OPTUNA, 20),
                       show_progress_bar=False)
        best_hp = study.best_params
        lr_opt = best_hp.pop("lr", 1e-3)
        print(f"  Mejor Kappa CV: {study.best_value:.4f} | Params: {best_hp}")
        best_hp["lr"] = lr_opt
        with open(ruta_hp, "w") as f:
            json.dump(best_hp, f)

    # ── Entrenamiento final ────────────────────────────────────────────────────
    lr_opt = best_hp.pop("lr", 1e-3)
    clf_final = TabNetClassifier(
        **{k: v for k, v in best_hp.items() if k != "lr"},
        optimizer_fn=torch.optim.Adam,
        optimizer_params={"lr": lr_opt},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={"step_size": 10, "gamma": 0.9},
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        cat_emb_dim=3,
        verbose=0,
        device_name=DISPOSITIVO_TABNET,
        seed=SEMILLA,
    )
    clf_final.fit(
        X_train_imp.astype(np.float32),
        y_train,
        max_epochs=200,
        patience=20,
        batch_size=1024,
        virtual_batch_size=128,
        weights=1,
    )

    y_pred = clf_final.predict(X_test_imp.astype(np.float32))
    y_prob = clf_final.predict_proba(X_test_imp.astype(np.float32))

    ruta_modelo = CARPETA_MODELS / f"{nombre_modelo}_{subperiodo}"
    clf_final.save_model(str(ruta_modelo))
    print(f"  Modelo guardado: {ruta_modelo.name}.zip")
    print("  ⚠ Limitación: TabNet usa pesos por clase (no sample_weight individual).")
    print("    Ver análisis de sensibilidad regional en sección de resultados.")

    metricas = evaluar_modelo(y_test, y_pred, y_prob, nombre_modelo, subperiodo)
    RESULTADOS_GLOBALES.append(metricas)
    return metricas


print("✓ Función de entrenamiento TabNet definida.")

## 18. Ciclo principal de entrenamiento

Itera sobre los tres subperiodos y entrena los cinco modelos en cada uno,
aplicando el preprocesamiento diferenciado correspondiente.

In [ ]:
# =============================================================================
# CELDA 18 — CICLO PRINCIPAL DE ENTRENAMIENTO
# Itera SP1 → SP2 → SP3, entrena los 5 modelos, guarda resultados.
# =============================================================================

for sp_nombre, sp_cfg in SUBPERIODOS.items():
    print()
    print("#" * 65)
    print(f"# SUBPERIODO: {sp_nombre} — {sp_cfg['descripcion']}")
    print("#" * 65)

    # ── Construir splits ───────────────────────────────────────────────────────
    X_train_trees, y_train, X_test_trees, y_test, w_train, w_test = construir_split(
        df, sp_nombre, FEATURES_TREES
    )
    X_train_olo, y_train_olo, X_test_olo, y_test_olo, w_train_olo, _ = construir_split(
        df, sp_nombre, FEATURES_OLO
    )
    resumen_split(sp_nombre, X_train_trees, y_train, X_test_trees, y_test)

    # ── Imputación para OLO y TabNet ──────────────────────────────────────────
    print("\n  [Imputación MICE para OLO/TabNet...]")
    X_train_olo_imp, X_test_olo_imp, imp_olo = imputar_para_olo_tabnet(
        X_train_olo, X_test_olo
    )
    X_train_tab_imp, X_test_tab_imp, imp_tab = imputar_para_olo_tabnet(
        X_train_trees, X_test_trees
    )

    # ── Normalización ──────────────────────────────────────────────────────────
    print("  [Normalización...]")
    X_train_olo_sc, X_test_olo_sc, sc_olo = normalizar(
        X_train_olo_imp, X_test_olo_imp, metodo="standard"
    )
    X_train_tab_sc, X_test_tab_sc, sc_tab = normalizar(
        X_train_tab_imp, X_test_tab_imp, metodo="minmax"
    )

    # ── Índices y dimensiones de categóricas para TabNet ──────────────────────
    cols_tabnet = list(X_train_tab_sc.columns)
    cat_idxs_tab = [i for i, c in enumerate(cols_tabnet) if c in VARS_CATEGORICAS]
    cat_dims_tab = []
    for col in [c for c in VARS_CATEGORICAS if c in cols_tabnet]:
        n_cats = int(X_train_tab_sc[col].nunique()) + 1  # +1 para valores desconocidos
        cat_dims_tab.append(n_cats)

    # ── Guardar datasets procesados en Parquet ─────────────────────────────────
    # Parquet preserva dtypes, es 5-10x más rápido que CSV y ocupa ~80% menos.
    print("\n  [Guardando datasets en Parquet...]")
    X_train_trees.assign(target=y_train.values).to_parquet(
        CARPETA_PROC / f"{sp_nombre}_train.parquet", index=False
    )
    X_test_trees.assign(target=y_test.values).to_parquet(
        CARPETA_PROC / f"{sp_nombre}_test.parquet", index=False
    )

    # Guardar imputers y scalers para reproducibilidad
    joblib.dump(imp_olo, CARPETA_MODELS / f"imputer_olo_{sp_nombre}.pkl")
    joblib.dump(imp_tab, CARPETA_MODELS / f"imputer_tab_{sp_nombre}.pkl")
    joblib.dump(sc_olo,  CARPETA_MODELS / f"scaler_olo_{sp_nombre}.pkl")
    joblib.dump(sc_tab,  CARPETA_MODELS / f"scaler_tab_{sp_nombre}.pkl")
    print(f"  Datasets y preprocesadores guardados en {CARPETA_PROC.name}/")

    # ── Entrenamiento de los 5 modelos ─────────────────────────────────────────
    # Modelo 1: OLO
    entrenar_olo(
        X_train_olo_sc.values, y_train_olo.values,
        X_test_olo_sc.values, y_test_olo.values,
        w_train_olo, sp_nombre,
    )

    # Modelo 2: XGBoost
    entrenar_xgboost(
        X_train_trees, y_train, X_test_trees, y_test,
        w_train, sp_nombre,
    )

    # Modelo 3: CatBoost
    entrenar_catboost(
        X_train_trees, y_train, X_test_trees, y_test,
        w_train, sp_nombre,
    )

    # Modelo 4: LightGBM
    entrenar_lightgbm(
        X_train_trees, y_train, X_test_trees, y_test,
        w_train, sp_nombre,
    )

    # Modelo 5: TabNet
    entrenar_tabnet(
        X_train_tab_sc.values, y_train.values,
        X_test_tab_sc.values, y_test.values,
        sp_nombre, cat_idxs_tab, cat_dims_tab,
    )

    gc.collect()
    print(f"\n✓ {sp_nombre} completado.")

print()
print("=" * 65)
print("✅ CICLO COMPLETO DE ENTRENAMIENTO FINALIZADO")
print("=" * 65)

## 19. Tabla comparativa de resultados

In [ ]:
# =============================================================================
# CELDA 19 — Tabla comparativa de métricas por modelo y subperiodo
# =============================================================================

df_resultados = pd.DataFrame(RESULTADOS_GLOBALES)

# Ordenar por subperiodo y métrica principal
df_resultados = df_resultados.sort_values(["subperiodo", "kappa_cuadratico"],
                                           ascending=[True, False])

# Formatear para visualización
cols_metricas = [
    "modelo", "subperiodo", "balanced_accuracy",
    "f1_macro", "f1_weighted",
    "kappa_lineal", "kappa_cuadratico",
    "mae_ordinal", "auroc_macro"
]
df_display = df_resultados[[c for c in cols_metricas if c in df_resultados.columns]]

print("Tabla comparativa de métricas:")
print(df_display.to_string(index=False, float_format="{:.4f}".format))

# Guardar resultados
ruta_resultados = CARPETA_RESULTS / "resultados_modelos.csv"
df_resultados.to_csv(ruta_resultados, index=False)
print(f"\n✓ Resultados guardados: {ruta_resultados}")

# Guardar también en Parquet para uso en notebooks de XAI
df_resultados.to_parquet(
    CARPETA_RESULTS / "resultados_modelos.parquet", index=False
)

# Pivot: modelo × subperiodo para Kappa cuadrático
print("\nKappa Cuadrático por modelo y subperiodo:")
pivot = df_resultados.pivot(
    index="modelo", columns="subperiodo", values="kappa_cuadratico"
)
print(pivot.to_string(float_format="{:.4f}".format))

## 20. Visualización comparativa de rendimiento

In [ ]:
# =============================================================================
# CELDA 20 — Visualización: rendimiento por modelo y subperiodo
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    "Rendimiento comparativo de modelos por subperiodo\n"    "(Expanding Window Walk-Forward Validation)",
    fontsize=13, fontweight="bold", y=1.02
)

metricas_plot = [
    ("kappa_cuadratico", "Kappa Cuadrático"),
    ("f1_macro",         "F1 Macro"),
    ("mae_ordinal",      "MAE Ordinal (↓ mejor)"),
]

colores = {
    "OLO"    : "#2196F3",
    "XGBoost": "#4CAF50",
    "CatBoost": "#FF9800",
    "LightGBM": "#9C27B0",
    "TabNet" : "#F44336",
}

for ax, (metrica, titulo) in zip(axes, metricas_plot):
    for modelo in ["OLO", "XGBoost", "CatBoost", "LightGBM", "TabNet"]:
        sub = df_resultados[df_resultados["modelo"] == modelo]
        if sub.empty or metrica not in sub.columns:
            continue
        ax.plot(
            sub["subperiodo"], sub[metrica],
            marker="o", linewidth=2, markersize=7,
            label=modelo, color=colores.get(modelo, "gray")
        )
    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel("Subperiodo")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    if metrica == "mae_ordinal":
        ax.invert_yaxis()   # MAE más bajo = mejor rendimiento

plt.tight_layout()
ruta_fig = CARPETA_RESULTS / "rendimiento_comparativo.png"
plt.savefig(ruta_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Figura guardada: {ruta_fig.name}")

## 21. Registro de versiones y entorno (reproducibilidad)

In [ ]:
# =============================================================================
# CELDA 21 — Registro de versiones para reproducibilidad
# =============================================================================
import importlib, platform, sys

librerias = [
    "pandas", "numpy", "sklearn", "xgboost", "lightgbm",
    "catboost", "pytorch_tabnet", "optuna", "torch", "joblib",
]

print("Entorno de ejecución:")
print(f"  Fecha          : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Python         : {sys.version.split()[0]}")
print(f"  Sistema        : {platform.system()} {platform.release()}")
print(f"  Semilla global : {SEMILLA}")
print()
print("Versiones de librerías:")
for lib in librerias:
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, "__version__", "N/D")
        print(f"  {lib:<20}: {ver}")
    except ImportError:
        print(f"  {lib:<20}: NO INSTALADA")

## Guardar el pipeline completo

In [ ]:
# Al final del entrenamiento, encapsular todo en un artefacto único
from sklearn.pipeline import Pipeline
import joblib

artefacto_modelo = {
    # El modelo entrenado
    "modelo": clf_final,
    
    # Transformaciones aprendidas (deben cargarse, no reimplementarse)
    "imputer": imputer_ajustado,       # IterativeImputer ya ajustado
    "scaler": scaler_ajustado,         # StandardScaler o MinMaxScaler ya ajustado
    
    # Metadata del pipeline (para validación en producción)
    "features_esperadas": FEATURES_TREES,   # lista ordenada de columnas
    "version_pipeline": "1.0.0",
    "subperiodo": sp_nombre,
    "fecha_entrenamiento": datetime.now().isoformat(),
    
    # Parámetros de las transformaciones deterministas
    # (documentados, no calculados)
    "transformaciones": {
        "invertir_likert_4": VARS_LB_INVERTIR_LIKERT_4,
        "invertir_vdem": VARS_VDEM_INVERTIR,
        "recodificaciones_binarias": {
            "B_006_061": {1: 1, 2: 0},
            "B_001_101": {1: 1, 2: 0},
            "H_001_011": {1: 1, 2: 0},
            "S_001": {1: 0, 2: 1},
        },
        "nsnr_a_nan": NSNR_GENERICOS,
        "valores_especiales_nan": {"A_007_071": [97], "C_003_003_011": [5]},
    }
}

joblib.dump(artefacto_modelo, CARPETA_MODELS / f"pipeline_completo_{sp_nombre}.pkl")

## Para carar en Producción

In [ ]:
artefacto = joblib.load("models/pipeline_completo_SP3.pkl")

def predecir(datos_crudos: dict) -> dict:
    """
    Recibe los valores ORIGINALES del cuestionario (tal como los respondió 
    el encuestado) y devuelve la predicción de satisfacción democrática.
    
    datos_crudos: diccionario con los valores originales sin transformar.
    """
    df = pd.DataFrame([datos_crudos])
    
    # 1. Aplicar transformaciones deterministas (orden crítico)
    df = aplicar_transformaciones_deterministas(df, artefacto["transformaciones"])
    
    # 2. Asegurar que las columnas están en el orden correcto
    df = df[artefacto["features_esperadas"]]
    
    # 3. Aplicar transformaciones aprendidas
    df_imputed = artefacto["imputer"].transform(df)      # solo .transform()
    df_scaled  = artefacto["scaler"].transform(df_imputed)  # solo .transform()
    
    # 4. Predecir
    clase_pred = artefacto["modelo"].predict(df_scaled)[0]
    probabilidades = artefacto["modelo"].predict_proba(df_scaled)[0]
    
    # 5. Traducir la clase base-0 a etiqueta legible
    etiquetas = {
        0: "Para nada satisfecho",
        1: "No muy satisfecho", 
        2: "Más bien satisfecho",
        3: "Muy satisfecho"
    }
    
    return {
        "clase_predicha": int(clase_pred),
        "etiqueta": etiquetas[int(clase_pred)],
        "probabilidades": {etiquetas[i]: float(p) for i, p in enumerate(probabilidades)}
    }